In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
# Create SparkSession
spark =  SparkSession.builder \
                    .master("spark://spark-master:7077") \
                    .appName("example") \
                    .config("spark.executor.memory", "1g") \
                    .getOrCreate()
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/11 07:30:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
# # Create SparkSession
# spark = SparkSession.builder \
#     .appName("KafkaStream") \
#     .config("spark.jars.packages", 
#             "org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.0") \
#     .getOrCreate()

# # spark

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [4]:
schema = StructType([
    StructField("LocationID", IntegerType(), True),
    StructField("Borough", StringType(), True),
    StructField("Zone", StringType(), True),
    StructField("service_zone", StringType(), True)
])

In [5]:
df = spark \
    .readStream \
    .option("header", "true") \
    .option("maxFilesPerTrigger", 1) \
    .schema(schema) \
    .csv("/app/streamfolder/")

In [6]:
processed = df.filter(df["LocationID"] > 10)

In [7]:
# query = df_stream.writeStream
#             .format("delta")
#             .trigger(availableNow=True)
#             .option("checkpointLocation", checkpoint_path)
#             .foreachBatch(lambda df, epoch_id: process_data(df, epoch_id))
#             .start()

IndentationError: unexpected indent (2170570731.py, line 2)

In [8]:
query = processed \
    .writeStream \
    .outputMode("append") \
    .format("csv") \
    .option("header", "true") \
    .option("path", "/app/streamoutput/") \
    .option("checkpointLocation", "/app/checkpoint/") \
    .trigger(processingTime="10 seconds") \
    .start()

26/03/11 07:31:13 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/03/11 07:31:20 WARN TaskSetManager: Lost task 0.0 in stage 0.0 (TID 0) (172.18.0.2 executor 1): org.apache.spark.SparkException: [FAILED_READ_FILE.FILE_NOT_EXIST] Encountered error while reading file file:///app/streamfolder/taxi_zone_lookup.csv. File does not exist. It is possible the underlying files have been updated.
You can explicitly invalidate the cache in Spark by running 'REFRESH TABLE tableName' command in SQL or by recreating the Dataset/DataFrame involved. SQLSTATE: KD001
	at org.apache.spark.sql.errors.QueryExecutionErrors$.fileNotExistError(QueryExecutionErrors.scala:873)
	at org.apache.spark.sql.execution.datasources.v2.FileDataSourceV2$.attachFilePath(FileDataSourceV2.scala:140)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:141)
	at scala.collection.Iterator$$anon$9.hasNext(Itera

In [14]:
processed.show()

AnalysisException: Queries with streaming sources must be executed with writeStream.start(), or from a streaming table or flow definition within a Spark Declarative Pipeline.;
FileSource[/app/streamfolder/]

In [7]:
csv_df=spark.read.csv("/app/data/taxi_zone_lookup.csv",header=True,inferSchema=True)

In [8]:
csv_df.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when

In [2]:
spark = SparkSession.builder.appName("PracticeStream").getOrCreate()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/10 11:02:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# Generates rows automatically — no Kafka, no setup!
df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 10) \
    .load()

In [4]:
result = df.select(
    col("timestamp"),
    col("value"),
    (col("value") * 2).alias("doubled"),
    when(col("value") % 2 == 0, "even")
    .otherwise("odd").alias("parity")
)

In [5]:
result.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", False) \
    .trigger(processingTime="2 seconds") \
    .start() \
    .awaitTermination()

26/03/10 11:02:34 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-436b5253-b918-4e3a-8439-71afd3298617. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/03/10 11:02:34 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+---------+-----+-------+------+
|timestamp|value|doubled|parity|
+---------+-----+-------+------+
+---------+-----+-------+------+

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------------------+-----+-------+------+
|timestamp              |value|doubled|parity|
+-----------------------+-----+-------+------+
|2026-03-10 11:02:35.209|0    |0      |even  |
|2026-03-10 11:02:36.009|8    |16     |even  |
|2026-03-10 11:02:35.309|1    |2      |odd   |
|2026-03-10 11:02:36.109|9    |18     |odd   |
|2026-03-10 11:02:35.409|2    |4      |even  |
|2026-03-10 11:02:35.509|3    |6      |odd   |
|2026-03-10 11:02:35.609|4    |8      |even  |
|2026-03-10 11:02:35.709|5    |10     |odd   |
|2026-03-10 11:02:35.809|6    |12     |even  |
|2026-03-10 11:02:35.909|7    |14     |odd   |
+-----------------------+-----+-------+------+

--------------

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/opt/bitnami/spark/python/lib/py4j.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/bitnami/spark/python/lib/py4j.zip/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/bitnami/python/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

-------------------------------------------
Batch: 22
-------------------------------------------
+-----------------------+-----+-------+------+
|timestamp              |value|doubled|parity|
+-----------------------+-----+-------+------+
|2026-03-10 11:03:15.209|400  |800    |even  |
|2026-03-10 11:03:16.009|408  |816    |even  |
|2026-03-10 11:03:16.809|416  |832    |even  |
|2026-03-10 11:03:15.309|401  |802    |odd   |
|2026-03-10 11:03:16.109|409  |818    |odd   |
|2026-03-10 11:03:16.909|417  |834    |odd   |
|2026-03-10 11:03:15.409|402  |804    |even  |
|2026-03-10 11:03:16.209|410  |820    |even  |
|2026-03-10 11:03:17.009|418  |836    |even  |
|2026-03-10 11:03:15.509|403  |806    |odd   |
|2026-03-10 11:03:16.309|411  |822    |odd   |
|2026-03-10 11:03:17.109|419  |838    |odd   |
|2026-03-10 11:03:15.609|404  |808    |even  |
|2026-03-10 11:03:16.409|412  |824    |even  |
|2026-03-10 11:03:15.709|405  |810    |odd   |
|2026-03-10 11:03:16.509|413  |826    |odd   |
|2026-03-

-------------------------------------------
Batch: 438
-------------------------------------------
+-----------------------+-----+-------+------+
|timestamp              |value|doubled|parity|
+-----------------------+-----+-------+------+
|2026-03-10 11:17:07.209|8720 |17440  |even  |
|2026-03-10 11:17:08.009|8728 |17456  |even  |
|2026-03-10 11:17:08.809|8736 |17472  |even  |
|2026-03-10 11:17:07.309|8721 |17442  |odd   |
|2026-03-10 11:17:08.109|8729 |17458  |odd   |
|2026-03-10 11:17:08.909|8737 |17474  |odd   |
|2026-03-10 11:17:07.409|8722 |17444  |even  |
|2026-03-10 11:17:08.209|8730 |17460  |even  |
|2026-03-10 11:17:09.009|8738 |17476  |even  |
|2026-03-10 11:17:07.509|8723 |17446  |odd   |
|2026-03-10 11:17:08.309|8731 |17462  |odd   |
|2026-03-10 11:17:09.109|8739 |17478  |odd   |
|2026-03-10 11:17:07.609|8724 |17448  |even  |
|2026-03-10 11:17:08.409|8732 |17464  |even  |
|2026-03-10 11:17:07.709|8725 |17450  |odd   |
|2026-03-10 11:17:08.509|8733 |17466  |odd   |
|2026-03

-------------------------------------------
Batch: 440
-------------------------------------------
+-----------------------+-----+-------+------+
|timestamp              |value|doubled|parity|
+-----------------------+-----+-------+------+
|2026-03-10 11:17:11.209|8760 |17520  |even  |
|2026-03-10 11:17:12.009|8768 |17536  |even  |
|2026-03-10 11:17:12.809|8776 |17552  |even  |
|2026-03-10 11:17:11.309|8761 |17522  |odd   |
|2026-03-10 11:17:12.109|8769 |17538  |odd   |
|2026-03-10 11:17:12.909|8777 |17554  |odd   |
|2026-03-10 11:17:11.409|8762 |17524  |even  |
|2026-03-10 11:17:12.209|8770 |17540  |even  |
|2026-03-10 11:17:13.009|8778 |17556  |even  |
|2026-03-10 11:17:11.509|8763 |17526  |odd   |
|2026-03-10 11:17:12.309|8771 |17542  |odd   |
|2026-03-10 11:17:13.109|8779 |17558  |odd   |
|2026-03-10 11:17:11.609|8764 |17528  |even  |
|2026-03-10 11:17:12.409|8772 |17544  |even  |
|2026-03-10 11:17:11.709|8765 |17530  |odd   |
|2026-03-10 11:17:12.509|8773 |17546  |odd   |
|2026-03